In [ ]:
%pip install sammo pandas --upgrade

### Prompt Engineering

You can easily a/b test prompts in Sammo by swapping out sections of the metaprompt.

In [ ]:
import getpass
import os
from pathlib import Path
import pandas as pd
import sammo
from sammo.base import EvaluationScore
from sammo.components import Output
from sammo.data import DataTable
from sammo.runners import OpenAIChat

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

_ = sammo.setup_logger("WARNING")  # we're only interested in warnings for now

BASE_DIR = Path.cwd()
if not (BASE_DIR / "transaction_data_with_classifications.csv").exists():
    BASE_DIR = BASE_DIR / "prompt_optimization_and_evals"

MODEL = os.getenv("SAMMO_MODEL", "gpt-4o")
# SAMMO 0.3.3 sends max_tokens=None when this is omitted.
MAX_OUTPUT_TOKENS = int(os.getenv("SAMMO_MAX_OUTPUT_TOKENS", "1000"))
CACHE_FILE = str(BASE_DIR / os.getenv("CACHE_FILE", "cache.tsv"))

In [ ]:


runner = OpenAIChat(
    model_id=MODEL,
    api_config={"api_key": os.environ["OPENAI_API_KEY"]},
    cache=CACHE_FILE,
    timeout=30,
    max_context_window=MAX_OUTPUT_TOKENS,
)

def load_data():
    df = pd.read_csv(BASE_DIR / "transaction_data_with_classifications.csv")
    mydata = DataTable.from_pandas(df, input_fields="description", output_fields="classification", constants={"instructions": "Determine how to classify these transactions."})
    return mydata

def accuracy(y_true: DataTable, y_pred: DataTable) -> EvaluationScore:
    y_true = y_true.outputs.values
    y_pred = y_pred.outputs.values
    n_correct = sum([y_p == y_t for y_p, y_t in zip(y_pred, y_true)])
    return EvaluationScore(n_correct / len(y_true))

from sammo.instructions import MetaPrompt, Section, Paragraph, InputData, FewshotExamples
from sammo.dataformatters import (
    QuestionAnswerFormatter,
)

mydata = load_data()
sample = mydata.sample(20, seed=42)

labels = ["Rent", "Bills", "Other", "Food", "Entertainment", "Utilities", "Salary", "Taxes", "Insurance", "Unknown"]

mprompt = MetaPrompt(
    [
        Section("Instructions", mydata.constants["instructions"]),
        Section("Examples", FewshotExamples(mydata.sample(3, seed=43))),
        Paragraph(f"\nOutput labels: {', '.join(labels)}"),
        Paragraph(InputData()),
    ],
    render_as="markdown",
    data_formatter=QuestionAnswerFormatter(labels),
)
# automatically wraps it with the right parser component
mprompt_parsed = mprompt.with_extractor("empty_result")

result = Output(mprompt_parsed, minibatch_size=5, on_error="empty_result").run(
    runner, sample
)
result[:5]


In [ ]:
import getpass
import os
from pathlib import Path
import pandas as pd
import sammo
from sammo.base import EvaluationScore
from sammo.components import Output
from sammo.data import DataTable
from sammo.runners import OpenAIChat

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

_ = sammo.setup_logger("WARNING")  # we're only interested in warnings for now

BASE_DIR = Path.cwd()
if not (BASE_DIR / "transaction_data_with_classifications.csv").exists():
    BASE_DIR = BASE_DIR / "prompt_optimization_and_evals"

MODEL = os.getenv("SAMMO_MODEL", "gpt-4o")
# SAMMO 0.3.3 sends max_tokens=None when this is omitted.
MAX_OUTPUT_TOKENS = int(os.getenv("SAMMO_MAX_OUTPUT_TOKENS", "1000"))
CACHE_FILE = str(BASE_DIR / os.getenv("CACHE_FILE", "cache.tsv"))

runner = OpenAIChat(
    model_id=MODEL,
    api_config={"api_key": os.environ["OPENAI_API_KEY"]},
    cache=CACHE_FILE,
    timeout=30,
    max_context_window=MAX_OUTPUT_TOKENS,
)

def load_data():
    df = pd.read_csv(BASE_DIR / "transaction_data_with_classifications.csv")
    mydata = DataTable.from_pandas(df, input_fields="description", output_fields="classification", constants={"instructions": "Determine how to classify these transactions."})
    return mydata

def accuracy(y_true: DataTable, y_pred: DataTable) -> EvaluationScore:
    y_true = y_true.outputs.values
    y_pred = y_pred.outputs.values
    n_correct = sum([y_p == y_t for y_p, y_t in zip(y_pred, y_true)])
    return EvaluationScore(n_correct / len(y_true))

from sammo.instructions import MetaPrompt, Section, Paragraph, InputData, FewshotExamples
from sammo.dataformatters import (
    QuestionAnswerFormatter,
)

mydata = load_data()
sample = mydata.sample(20, seed=42)

labels = ["Rent", "Bills", "Other", "Food", "Entertainment", "Utilities", "Salary", "Taxes", "Insurance", "Unknown"]

mprompt = MetaPrompt(
    [
        Section("Instructions", mydata.constants["instructions"]),
        Section("Examples", FewshotExamples(mydata.sample(3, seed=43))),
        Paragraph(f"\nOutput labels: {', '.join(labels)}"),
        Paragraph(InputData()),
    ],
    render_as="markdown",
    data_formatter=QuestionAnswerFormatter(labels),
)
# automatically wraps it with the right parser component
mprompt_parsed = mprompt.with_extractor("empty_result")

result = Output(mprompt_parsed, minibatch_size=5, on_error="empty_result").run(
    runner, sample
)
result[:5]


In [ ]:
accuracy(result, sample)

In [ ]:
from sammo.search import EnumerativeSearch
from sammo.search_op import one_of
from sammo.components import Output


def labeling_prompt_space():
    instructions = one_of(
        [
            "Determine how to classify these transactions.",
            "Determine how to classify these transactions. THIS IS VERY IMPORTANT TO MY CAREER.",
        ]
    )
    
    mprompt = MetaPrompt(
        [
            Section("Instructions", instructions),
            Section("Examples", FewshotExamples(mydata.sample(3, seed=43))),
            Paragraph(f"\nOutput labels: {', '.join(labels)}"),
            Paragraph(InputData()),
        ],
        render_as="markdown",
        data_formatter=QuestionAnswerFormatter(labels),
    )
    # automatically wraps it with the right parser component
    mprompt_parsed = mprompt.with_extractor("empty_result")

    return Output(mprompt_parsed, minibatch_size=5, on_error="empty_result")


In [ ]:
sample = mydata.sample(25, seed=42)
searcher = EnumerativeSearch(runner, labeling_prompt_space, accuracy)
y_pred = searcher.fit_transform(sample)
searcher.show_report()

### Prompt Optimization

Sammo also provides tools for optimizing prompts automatically.

In [ ]:
from sammo.dataformatters import PlainFormatter

class InititialCandidates:
    def __init__(self, dtrain):
        self.dtrain = dtrain

    def __call__(self):
        example_formatter = PlainFormatter(
            all_labels=self.dtrain.outputs.unique(), orient="item"
        )

        labels = self.dtrain.outputs.unique()
        instructions = MetaPrompt(
            [
                Paragraph("Instructions: "),
                Paragraph(
                    one_of(
                        [
                            self.dtrain.constants["instructions"],
                            "",
                            "Find the best output label given the input.",
                            self.dtrain.constants["instructions"] * 2,
                        ]
                    )
                ),
                Paragraph("\n"),
                Paragraph(
                    f"Output labels: {', '.join(labels)}\n" if len(labels) <= 10 else ""
                ),
                Paragraph(InputData()),
                Paragraph("Output: "),
            ],
            render_as="raw",
            data_formatter=example_formatter,
        )

        return Output(
            instructions.with_extractor("raise"),
            minibatch_size=1,
            on_error="empty_result",
        )

In [ ]:
from sammo.mutators import BagOfMutators, InduceInstructions, Paraphrase

mydata = load_data()
d_train = mydata.sample(10, seed=4)

mutation_operators = BagOfMutators(
    InititialCandidates(d_train),
    InduceInstructions("#instructions", d_train),
    Paraphrase("#instructions"),
    sample_for_init_candidates=False,
)

In [ ]:
from sammo.search import BeamSearch

prompt_optimizer = BeamSearch(
            runner,
            mutation_operators,
            accuracy,
            maximize=True,
            depth=3,
            mutations_per_beam=2,
            n_initial_candidates=4,
            beam_width=4,
            add_previous=True,
    )
prompt_optimizer.fit(d_train)
prompt_optimizer.show_report()

In [ ]:
prompt_optimizer.show_report()

In [ ]:
print(prompt_optimizer.best_prompt)